#  Phase 3 - Data Cleaning & Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [5]:
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

**Check Missing Values**

In [6]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order Items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Reviews
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review

**Handle Missing Values**

In [7]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("Unknown")
)

In [8]:
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[dimension_columns] = (
    products[dimension_columns]
    .fillna(products[dimension_columns].median())
)

**Merge Product Categories**

In [9]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

**Building Master Dataset**

Merge Orders and Customers

In [10]:
master = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

Add Order Items

In [11]:
master = master.merge(
    order_items,
    on="order_id",
    how="left"
)

Add Products

In [12]:
master = master.merge(
    products,
    on="product_id",
    how="left"
)

Add Payments

In [13]:
master = master.merge(
    payments,
    on="order_id",
    how="left"
)

Add Reviews

In [14]:
master = master.merge(
    reviews,
    on="order_id",
    how="left"
)

**Feature Engineering**

Delivery Time (days)

In [15]:
master["delivery_days"] = (
    master["order_delivered_customer_date"] -
    master["order_purchase_timestamp"]
).dt.days

Approval Time (hours)

In [16]:
master["approval_hours"] = (
    master["order_approved_at"] -
    master["order_purchase_timestamp"]
).dt.total_seconds() / 3600

Delivery Delay

In [17]:
master["delivery_delay_days"] = (
    master["order_delivered_customer_date"] -
    master["order_estimated_delivery_date"]
).dt.days

Order Month

In [18]:
master["order_month"] = (
    master["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

Order Year

In [19]:
master["order_year"] = (
    master["order_purchase_timestamp"]
    .dt.year
)

**Verify the Master Dataset**

In [20]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,delivery_days,approval_hours,delivery_delay_days,order_month,order_year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,8.0,0.178333,-8.0,2017-10,2017
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,8.0,0.178333,-8.0,2017-10,2017
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,8.0,0.178333,-8.0,2017-10,2017
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,30.713889,-6.0,2018-07,2018
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,0.276111,-18.0,2018-08,2018


In [21]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  object        
 1   customer_id                    119143 non-null  object        
 2   order_status                   119143 non-null  object        
 3   order_purchase_timestamp       119143 non-null  datetime64[ns]
 4   order_approved_at              118966 non-null  datetime64[ns]
 5   order_delivered_carrier_date   117057 non-null  datetime64[ns]
 6   order_delivered_customer_date  115722 non-null  datetime64[ns]
 7   order_estimated_delivery_date  119143 non-null  datetime64[ns]
 8   customer_unique_id             119143 non-null  object        
 9   customer_zip_code_prefix       119143 non-null  int64         
 10  customer_city                  119143 non-null  object        
 11  

In [22]:
master.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,delivery_days,approval_hours,delivery_delay_days,order_month,order_year
count,119143,119143,119143,119143,118966,117057,115722,119143,119143,119143.000000,119143,119143,118310.000000,118310,118310,118310,118310.000000,118310.000000,118310,116601.000000,116601.000000,116601.000000,118310.000000,118310.000000,118310.000000,118310.000000,116576,119140.000000,119140,119140.000000,119140.000000,118146,118146.000000,13989,50245,118146,118146,115722.000000,118966.000000,115722.000000,119143,119143.000000
unique,99441,99441,8,NaN,NaN,NaN,NaN,NaN,96096,NaN,4119,27,NaN,32951,3095,93318,NaN,NaN,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71,NaN,5,NaN,NaN,98410,NaN,4527,36159,NaN,NaN,NaN,NaN,NaN,25,NaN
top,895ab968e7bb0d5659d16cd74cd1650c,270c23a11d024a44c896d1894b261a83,delivered,NaN,NaN,NaN,NaN,NaN,9a736b248f67d166d2fbb006bcb877c3,NaN,sao paulo,SP,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,4a3ca9315b744ce9f8e9374361493884,2017-08-14 20:43:31,NaN,NaN,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bed_bath_table,NaN,credit_card,NaN,NaN,eef5dbca8d37dfce6db7d7b16dd0525e,NaN,Recomendo,Muito bom,NaN,NaN,NaN,NaN,NaN,2017-11,NaN
freq,63,63,115723,NaN,NaN,NaN,NaN,NaN,75,NaN,18875,50265,NaN,536,2155,63,NaN,NaN,11988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11988,NaN,87776,NaN,NaN,63,NaN,494,259,NaN,NaN,NaN,NaN,NaN,9191,NaN
mean,NaN,NaN,NaN,2017-12-29 18:36:13.115760128,2017-12-30 04:49:18.425726976,2018-01-03 08:24:34.395524864,2018-01-12 20:55:38.199616,2018-01-22 15:21:10.241642240,NaN,35033.451298,NaN,NaN,1.196543,NaN,NaN,NaN,120.646603,20.032387,NaN,48.767498,785.967822,2.205161,2112.012002,30.264255,16.619094,23.074279,NaN,1.094737,NaN,2.941246,172.735135,NaN,4.015582,NaN,NaN,2018-01-11 13:17:50.103092992,2018-01-14 17:00:35.769302784,12.022589,10.581018,-12.048392,NaN,2017.535290
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,NaN,1003.000000,NaN,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,NaN,1.000000,NaN,0.000000,0.000000,NaN,1.000000,NaN,NaN,2016-10-02 00:00:00,2016-10-07 18:32:28,0.000000,0.000000,-147.000000,NaN,2016.000000
25%,NaN,NaN,NaN,2017-09-10 20:15:46,2017-09-11 15:50:48.500000,2017-09-14 19:52:12,2017-09-22 21:54:31.249999872,2017-10-02 00:00:00,NaN,11250.000000,NaN,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000,NaN,42.000000,346.000000,1.000000,300.000000,18.000000,8.000000,15.000000,NaN,1.000000,NaN,1.000000,60.850000,NaN,4.000000,NaN,NaN,2017-09-22 00:00:00,2017-09-25 11:15:40.750000128,6.000000,0.215556,-17.000000,NaN,2017.000000
50%,NaN,NaN,NaN,2018-01-17 11:59:12,2018-01-17 16:49:49,2018-01-23 17:03:08,2018-02-01 03:17:55,2018-02-14 00:00:00,NaN,24240.000000,NaN,NaN,1.000000,NaN,NaN,NaN,74.900000,16.280000,NaN,52.000000,600.000000,1.000000,700.000000,25.000000,13.000000,20.000000,NaN,1.000000,NaN,2.000000,108.160000,NaN,5.000000,NaN,NaN,2018-02-01 00:00:00,2018-02-03 12:04:23,10.000000,0.346944,-13.000000,NaN,2018.000000
75%,NaN,NaN,NaN,2018-05-03 13:18:30,2018-05-03 16:56:53,2018-05-07 14:57:00,2018-05-15 00:08:31.500000,2018-05-25 00:00:00,NaN,58475.000000,NaN,NaN,1.000000,NaN,NaN,NaN,134.900000,21.180000,NaN,57.000000,983.000000,3.000000,1800.000000,38.000000,20.000000,30.000000,NaN,1.000000,NaN,4.000000,189.240000,NaN,5.000000,NaN,NaN,2018-05-15 00:00:0

In [24]:
master.to_csv(
    "master_dataset.csv",
    index=False
)